# Chapter 6: B-Tree Variants
*From: Database Internals*

## TL;DR

- **Copy-on-Write B-Trees** — never mutate a page; copy the whole root-to-leaf path, atomically swap the root pointer. Attacks concurrency without latches; pays in space and CPU (LMDB).
- **Lazy B-Trees (WiredTiger, LA-Tree)** — buffer per-node (skiplist) or per-subtree updates in memory, reconcile lazily. Attacks write I/O and structural-modification cost; pays in read merge work.
- **FD-Trees** — append-only: small mutable head B-Tree plus logarithmic sorted runs stitched with fractional-cascading bridges. Attacks random writes on SSDs; pays read cost across runs.
- **Bw-Trees** — latch-free. Each logical node is a base page plus a CAS-installed delta chain, addressed through an in-memory mapping table. Attacks write amplification, space amplification, and latch complexity all at once.
- **Cache-Oblivious B-Trees** — van Emde Boas recursive layout over a packed array with gaps; asymptotically optimal block transfers without tuning any page/cache-line size. Mostly academic today.

## The Problem

Classical in-place B-Trees were designed for spinning disks where latency is dominated by the seek, so rewriting the same page in place is cheap once the head is there. SSDs flip the economics. A B-Tree update has to read the leaf page, mutate a few bytes, and write the whole page back; every small logical write becomes a full-page physical write. Structural modifications (splits and merges) propagate up the tree, each step also rewriting a full parent page. And because pages must reserve slack for future inserts, a lot of transferred bytes are empty.

The chapter frames this in the Bw-Tree section as three coupled problems: *"Write amplification is one of the most significant problems with in-place update implementations of B-Trees: subsequent updates to a B-Tree page may require updating a disk-resident page copy on every update. The second problem is space amplification: we reserve extra space to make updates possible. This also means that for each transferred useful byte carrying the requested data, we have to transfer some empty bytes and the rest of the page. The third problem is complexity in solving concurrency problems and dealing with latches."* Every variant below attacks some subset of those three.

## Write-Amplification Math

Before we taxonomize the variants, let's quantify what we are fighting. The numbers below use a 4KB page, 100-byte records, a fanout of 128, and a 70% leaf fill ratio — realistic for most classical B-Trees.

In [ ]:
# Write amplification: bytes written per logical record update
PAGE_SIZE = 4096           # bytes
RECORD_SIZE = 100          # bytes, key + value + overhead
FANOUT = 128               # internal-node fanout
FILL_RATIO = 0.70          # leaf pages are ~70% full
DELTA_HEADER = 24          # bytes: op-type, LSN, key-len, val-len, next-ptr
N_RECORDS = 10_000_000     # total records in the tree

# Derived tree shape
import math
records_per_leaf = int((PAGE_SIZE * FILL_RATIO) // RECORD_SIZE)
num_leaves = math.ceil(N_RECORDS / records_per_leaf)
height = max(1, math.ceil(math.log(num_leaves, FANOUT)) + 1)
print(f"records/leaf       = {records_per_leaf}")
print(f"num leaves         = {num_leaves:,}")
print(f"tree height        = {height}")
print()

# 1. In-place B-Tree: rewrite the one dirty leaf page
#    (assuming no split; splits would also rewrite the parent chain)
in_place_bytes = PAGE_SIZE
in_place_WA    = in_place_bytes / RECORD_SIZE

# 2. Copy-on-Write B-Tree (LMDB-style): copy every node on the root-to-leaf path
cow_bytes = PAGE_SIZE * height
cow_WA    = cow_bytes / RECORD_SIZE

# 3. Bw-Tree delta: append just the delta record, no page rewrite
bw_bytes = DELTA_HEADER + RECORD_SIZE
bw_WA    = bw_bytes / RECORD_SIZE

print(f"{'variant':<22}{'bytes/update':>14}{'write amp':>14}")
print(f"{'in-place B-Tree':<22}{in_place_bytes:>14}{in_place_WA:>14.1f}x")
print(f"{'copy-on-write':<22}{cow_bytes:>14}{cow_WA:>14.1f}x")
print(f"{'Bw-Tree delta':<22}{bw_bytes:>14}{bw_WA:>14.2f}x")
print()

# Space amplification: bytes held on disk per useful byte of record
in_place_space = 1.0 / FILL_RATIO           # slack inside leaves
cow_space      = 2.0 / FILL_RATIO           # two root versions + in-flight copies
bw_space_steady = 1.2                       # after consolidation; delta overhead

print(f"{'variant':<22}{'space amp':>12}")
print(f"{'in-place B-Tree':<22}{in_place_space:>12.2f}x")
print(f"{'copy-on-write':<22}{cow_space:>12.2f}x  (transient, until old root reclaimed)")
print(f"{'Bw-Tree steady':<22}{bw_space_steady:>12.2f}x")

The takeaway: the in-place tree writes 40x the useful bytes per update, COW writes roughly *height* times more than that, and the Bw-Tree delta approaches the theoretical minimum — at the cost of read-side delta-chain traversal.

## High-Level Taxonomy

```mermaid
graph TB
  Root["B-Tree Variants"]
  Root --> COW["Copy-on-Write<br/>copy root-to-leaf path,<br/>atomic root swap"]
  Root --> Lazy["Lazy B-Trees<br/>buffer updates per node<br/>or per subtree"]
  Root --> FD["FD-Trees<br/>head B-Tree + immutable<br/>cascaded sorted runs"]
  Root --> Bw["Bw-Trees<br/>base page + CAS delta chain<br/>via mapping table"]
  Root --> CO["Cache-Oblivious<br/>van Emde Boas layout<br/>over packed array"]

  COW -.-> LMDB["LMDB"]
  Lazy -.-> WT["WiredTiger / LA-Tree"]
  FD -.-> FlashDB["Flash-optimized stores"]
  Bw -.-> Hekaton["Hekaton / Sled / OpenBw-Tree"]
  CO -.-> Academic["mostly academic"]
```

## Deep Dive: Copy-on-Write B-Trees

Copy-on-write turns every write into a new tree version. When a page is about to be modified, its contents are copied, the copy is modified instead, and a parallel hierarchy is built up to the root. Because the root pointer is swung atomically only after all page writes have landed, readers never see a partially-updated tree and a crash in the middle of a write leaves the old version intact. This gives MVCC, crash safety, and lock-free reads essentially for free — writers don't block readers, and readers don't need latches at all.

The canonical implementation is LMDB: it holds only two root versions (current plus the one being committed), memory-maps the whole database so there is no separate page cache, and does not need a write-ahead log or checkpointing. During an update every branch node on the root-to-leaf path is copied — nodes actually touched by the update change, the rest are copied but structurally identical and point back to shared subtrees. Because the design is append-only and the old root is abandoned after commit, LMDB has no sibling pointers in leaves; range scans ascend back through the parent instead.

The trade is space and CPU: every logical update multiplies by tree height on the write path, and old pages must be kept alive until all readers viewing them finish. Since B-Trees are shallow (4-5 levels for billions of keys), this is usually acceptable.

```mermaid
graph TB
  subgraph Old["Old version (readers)"]
    OR["Root v1"]
    OI1["Internal A"]
    OI2["Internal B"]
    OL1["Leaf 1"]
    OL2["Leaf 2"]
    OL3["Leaf 3 (old)"]
    OR --> OI1
    OR --> OI2
    OI1 --> OL1
    OI1 --> OL2
    OI2 --> OL3
  end

  subgraph New["New version (writer)"]
    NR["Root v2 (new copy)"]
    NI2["Internal B' (new copy)"]
    NL3["Leaf 3' (new copy)"]
    NR --> OI1
    NR --> NI2
    NI2 --> NL3
  end

  Swap["Atomic CAS on root pointer"] -.-> NR
```

**Why it matters**
- Readers run with zero synchronization — no latches, no version checks.
- Crash safety comes from the atomic root swap, not a WAL.
- Cost is O(height) page copies per update; worth it because B-Trees are shallow.
- Natural MVCC — old versions are exactly what long-running readers need.

## Deep Dive: Lazy B-Trees and Buffering

Lazy B-Trees attack write amplification by not writing on every update. Instead, updates land in a lightweight in-memory buffer attached to the node (or group of nodes) and are reconciled to disk in batches. Reads become slightly more expensive — they must merge the on-disk base image with whatever is in the buffer — but subsequent writes to the same node amortize across one flush.

WiredTiger (MongoDB's default storage engine) attaches an update buffer per page, implemented as a skiplist for good concurrent write throughput. A clean page is just an index built from the disk image; a dirty page additionally carries that skiplist. Reads merge the two; background threads perform reconciliation, which materializes the skiplist into a new page image and, if the page has grown past the threshold, splits it. The critical win is that splits and merges happen off the read/write path — foreground operations never pay the structural-modification tax.

The LA-Tree (Lazy-Adaptive Tree) pushes the idea up: buffers are attached not to individual nodes but to subtrees, and they are *cascaded*. An insert hits the root's buffer; when that fills, it spills into the buffers of the next level down, and so on until the leaves actually change. When leaves do change, many sibling updates batch into a single run — splits and merges propagate upward in batches too, amortizing even structural-modification I/O.

```mermaid
graph TB
  subgraph WT["WiredTiger per-node buffer"]
    WTD["Dirty page"]
    WTD --> WTI["base image<br/>from disk"]
    WTD --> WTU["update buffer<br/>skiplist of deltas"]
    WTR["Read"] --> WTD
    WTR --> WTMerge["merge in memory"]
    WTRec["Background reconciliation"] --> WTD
  end

  subgraph LA["LA-Tree cascaded subtree buffers"]
    B0["Root buffer"]
    B1a["Subtree buffer"]
    B1b["Subtree buffer"]
    B2a["Leaf buffer"]
    B2b["Leaf buffer"]
    B2c["Leaf buffer"]
    B0 -- overflow --> B1a
    B0 -- overflow --> B1b
    B1a -- overflow --> B2a
    B1a -- overflow --> B2b
    B1b -- overflow --> B2c
  end
```

**Why it matters**
- Subsequent writes to the same node collapse into one disk write.
- WiredTiger pushes splits/merges to a background thread; foreground stays fast.
- LA-Tree batches structural modifications across sibling leaves in one propagation wave.
- Cost is read-side merge with the buffer — bounded by buffer size per node.

## Deep Dive: FD-Trees

FD-Trees take buffering further. Rather than buffer per-node inside a B-Tree, they restrict the mutable surface to a tiny *head* B-Tree, and everything below is a sequence of *immutable sorted runs* with geometrically growing sizes. When the head fills, it is sorted into run L1. When L1 exceeds its threshold, it is merged with L2 and the result replaces L2 — exactly like LSM-Tree compaction. This eliminates random writes from the bulk of the dataset: the only random-write surface is the small head tree that fits comfortably in cache.

The read problem is that a lookup may now have to probe multiple sorted runs. FD-Trees solve this with *fractional cascading*: they pull every Nth element from a lower run up into the higher run as a *bridge* (or *fence* in the FD-Tree paper). A binary search on the top array lands you near the key; the bridge pointer forwards you to a narrow slice of the next-level array instead of a full `log n` search there. Each additional level adds only constant work after the first, so the total lookup cost is `O(log n + k)` for k levels rather than `O(k log n)`.

Deletes become tombstones (the paper calls them *filter entries*): an insert at the head that says "this key is gone". The tombstone propagates down through compactions, shadowing data records for the same key at lower levels, and is finally discarded when it reaches the lowest run.

```mermaid
graph TB
  Head["Head B-Tree<br/>small, mutable, in RAM"]
  L1["L1: immutable sorted run<br/>size = k * head"]
  L2["L2: immutable sorted run<br/>size = k^2 * head"]
  L3["L3: immutable sorted run<br/>size = k^3 * head"]

  Head -- flush when full --> L1
  L1 -- merge when full --> L2
  L2 -- merge when full --> L3

  L1 -. bridge pointers .-> L2
  L2 -. bridge pointers .-> L3

  Read["Lookup"] --> Head
  Read --> L1
  Read --> L2
  Read --> L3
```

Let's walk through the literal example from the chapter. Three arrays A1, A2, A3 are stitched together by pulling every other element from the lower-indexed array into the higher one.

In [ ]:
# Fractional cascading: literal chapter example
A1 = [12, 24, 25, 30, 32, 34, 39]
A2 = [16, 22, 25, 26, 28, 30, 35]
A3 = [11, 16, 24, 26, 30]

def pull_every_other(lower):
    """Pull every other element from the lower array as bridge candidates."""
    return lower[::2]

bridges_A2_from_A3 = pull_every_other(A3)   # what A2 pulls up from A3
bridges_A1_from_A2 = pull_every_other(A2)   # what A1 pulls up from A2
print(f"A3                    = {A3}")
print(f"bridges A2<-A3 (pulled) = {bridges_A2_from_A3}")
print(f"A2                    = {A2}")
print(f"bridges A1<-A2 (pulled) = {bridges_A1_from_A2}")
print(f"A1                    = {A1}")
print()

# Augmented arrays: original + bridge entries kept in sorted order,
# each bridge carrying a pointer to its position in the lower array.
def augment(arr, bridges, lower):
    """Merge bridges into arr; each bridge records its index in `lower`."""
    entries = [(v, "own", None) for v in arr]
    for v in bridges:
        entries.append((v, "bridge", lower.index(v)))
    entries.sort(key=lambda e: e[0])
    return entries

A2_aug = augment(A2, bridges_A2_from_A3, A3)
A1_aug = augment(A1, bridges_A1_from_A2, A2)

def _key_of(e):
    """Extract the comparable key from either a raw int or an augmented tuple."""
    return e[0] if isinstance(e, tuple) else e

def bsearch_leftmost(entries, key):
    """Classic binary search, counting comparisons. Returns (pos, comparisons)."""
    lo, hi, cmps = 0, len(entries), 0
    while lo < hi:
        mid = (lo + hi) // 2
        cmps += 1
        if _key_of(entries[mid]) < key:
            lo = mid + 1
        else:
            hi = mid
    return lo, cmps

# Search for key=26 across all three levels.
key = 26
pos1, c1 = bsearch_leftmost(A1_aug, key)
print(f"search key={key} in A1 (augmented, len={len(A1_aug)}): pos={pos1}, cmps={c1}")

# In A1_aug the nearest bridge hint to A2 comes from the last bridge at or
# before pos1. That hint is an index directly into A2, so the next search
# is a tiny linear walk instead of a full binary search.
def nearest_bridge_before(entries, pos):
    for i in range(pos, -1, -1):
        if entries[i][1] == "bridge":
            return entries[i]
    return None

hint = nearest_bridge_before(A1_aug, pos1)
print(f"bridge hint into A2: {hint}")

# Narrow the A2 search to a small window starting at the hint.
start = hint[2] if hint else 0
window = A2_aug[start:start + 4]  # constant-size window, not log(|A2|)
pos2_rel, c2 = bsearch_leftmost(window, key)
pos2 = start + pos2_rel
print(f"search key={key} in A2 window {window}: pos={pos2}, cmps={c2}")

# Repeat into A3.
hint2 = nearest_bridge_before(A2_aug, pos2)
print(f"bridge hint into A3: {hint2}")
start3 = hint2[2] if hint2 else 0
window3 = A3[start3:start3 + 4]
pos3_rel, c3 = bsearch_leftmost(window3, key)
pos3 = start3 + pos3_rel
print(f"search key={key} in A3 window {window3}: pos={pos3}, cmps={c3}")
print()
print(f"total comparisons with cascading: {c1 + c2 + c3}")
print(f"without cascading (3 full bsearches): "
      f"{sum(bsearch_leftmost(a, key)[1] for a in (A1, A2, A3))}")

**Why it matters**
- Random-write surface is bounded to the tiny head tree.
- Compactions convert many small random writes into one big sequential one — SSD-friendly.
- Fractional cascading bridges turn `O(k log n)` k-level search into `O(log n + k)`.
- Deletes use tombstones that percolate down exactly like LSM tombstones.

## Deep Dive: Bw-Trees

The Bw-Tree attacks all three pain points — write amplification, space amplification, and latch complexity — with one idea: make the node a *logical* thing, not a physical page. A logical node is a base page on disk plus a linked list of delta records prepended to it. An update appends a new delta describing the change; the base page and existing deltas are never modified. Since deltas are tiny compared to pages, write amplification collapses. Since nothing reserves slack for future writes, space amplification in steady state is minimal.

Concurrency is handled by an in-memory *mapping table* indexed by logical node id, whose slots hold physical pointers to the latest delta (or base page if none). To install a new delta, a writer constructs the delta with its `next` pointer set to the slot's current value, then compare-and-swaps the slot to point at the new delta. If the CAS fails, another writer won — retry. Readers never block: they snapshot the slot, walk the chain, and apply deltas over the base. Writers never block readers.

Structural-modification operations (splits/merges) can no longer be a single atomic page rewrite, so they become multi-step protocols using typed deltas: a *split delta* installed on the splitting node publishes the midpoint key and pointer to the new sibling; a separate *parent-update* step rewires the parent. Any thread that encounters a half-finished SMO is required to finish it before proceeding — cooperative completion. Concurrent splits/merges on the same parent are prevented with an *abort delta* that acts like a write lock. Long delta chains eventually slow reads, so a consolidation process merges the chain into a fresh base page and updates the mapping table. Old memory is reclaimed via epoch-based reclamation — once every thread that could have seen the old version has exited its epoch, the memory is free.

```mermaid
graph LR
  MT["Mapping Table<br/>in memory"]
  MT -- "slot for node 42" --> D2["delta: insert k=50"]
  D2 -- "next" --> D1["delta: update k=30"]
  D1 -- "next" --> Base["Base page 42<br/>on disk"]

  W["Writer"] -. "1 read slot" .-> MT
  W -. "2 allocate delta D3" .-> D3["delta: delete k=30"]
  D3 -. "3 set D3.next to D2" .-> D2
  W -. "4 CAS slot from D2 to D3" .-> MT
```

**Why it matters**
- Writes are ~ delta-sized, not page-sized — minimal write amplification.
- Zero latches — CAS on a pointer is the only synchronization primitive.
- SMOs are cooperative multi-step protocols, not atomic page rewrites.
- Consolidation + epoch-based reclamation are mandatory, not optional.

## Deep Dive: Cache-Oblivious B-Trees

Cache-oblivious B-Trees ask: what if the algorithm didn't need to know the page size, cache-line size, or block size at all, yet still achieved optimal block-transfer cost for *every* level of the memory hierarchy simultaneously? The answer is the van Emde Boas layout over a packed array.

Take a static binary tree and split it at the middle level of edges: top half is a small tree of height h/2, bottom half is a forest of small trees of height h/2. Lay the top subtree contiguously in memory, then each bottom subtree contiguously after it, and recurse into each piece. The key property is that any recursive subtree of size s is stored in contiguous memory of size s. So whatever block size the underlying hardware uses, once you load any block you get a full recursive subtree worth of nodes, and the asymptotic number of block transfers matches a cache-aware B-Tree tuned to that exact block size — but you didn't tune anything.

To make it dynamic (support inserts/deletes without a full relayout), the leaves live in a *packed array*: a contiguous run of memory with gaps spaced by a density threshold. Inserts shift a small neighborhood to open a slot; when density drifts outside its threshold, the local range is rebuilt. The vEB static tree above acts as an index into this packed array and is patched as the array rearranges. In practice, despite the elegant asymptotics, these structures have stayed academic: block transfers are the same O-bound as a cache-aware B-Tree, and real systems find that paging and eviction behavior dominate the constants.

```mermaid
graph TB
  subgraph Logical["Logical tree (height h)"]
    R["root"]
    T1["top subtree<br/>height h/2"]
    B1["bottom subtree<br/>height h/2"]
    B2["bottom subtree<br/>height h/2"]
    B3["bottom subtree<br/>height h/2"]
    B4["bottom subtree<br/>height h/2"]
    R --> T1
    T1 --> B1
    T1 --> B2
    T1 --> B3
    T1 --> B4
  end

  subgraph Physical["Contiguous block layout"]
    P["[ top subtree | B1 | B2 | B3 | B4 ]<br/>each recursive subtree is one contiguous run"]
  end

  Logical --> Physical
```

**Why it matters**
- One layout is optimal for every block size in the hierarchy — no tuning.
- Packed array with gaps makes inserts local; rebuilds are amortized.
- Same asymptotic block-transfer cost as a cache-aware B-Tree — so the constants decide.
- Mostly academic; may become relevant for byte-addressable NVM where page/block boundaries blur.

## Trade-offs & Alternatives

| Variant | Write Amp | Space Amp | Read Cost | Concurrency Model | Representative System |
|---|---|---|---|---|---|
| In-place B-Tree | High (full page per update) | ~1.4x (fill-ratio slack) | 1 I/O per level | Latches / page locks | PostgreSQL, InnoDB |
| Copy-on-Write | Very high (height x page) | ~2x (old root retained until readers drain) | 1 I/O per level | Lock-free reads, MVCC | LMDB |
| Lazy B-Tree | Medium (flush amortized) | Moderate (buffer overhead) | Base page + buffer merge | Fine-grained, background reconciliation | WiredTiger, LA-Tree |
| FD-Tree | Low (append to head, sequential compactions) | Moderate (immutable runs accumulate until merge) | k levels with cascading bridges | Mutable head only; runs immutable | Flash-oriented indexes |
| Bw-Tree | Very low (delta-sized writes) | Low in steady state (no slack reserved) | Base + delta-chain traversal | Latch-free via CAS on mapping table | Hekaton, Sled, OpenBw-Tree |
| Cache-Oblivious | Similar to in-place | Moderate (packed-array gaps) | Optimal block transfers, no tuning | Typically single-writer (academic) | Research prototypes |

## Key Takeaways

1. **Classical B-Trees are SSD-hostile** — they amplify writes by a full page per record update and reserve slack for space amplification. Every variant in this chapter attacks some slice of that.
2. **Reach for Copy-on-Write** when you want MVCC, crash safety, and latch-free reads and your write volume is modest (LMDB). The O(height) copy per update is acceptable because trees are shallow.
3. **Reach for a Lazy B-Tree** (WiredTiger-style) when you want to keep a B-Tree API and get most of the write-amplification win from buffering, without rewriting the world. Great default for OLTP engines on SSD.
4. **Reach for an FD-Tree** when writes dominate and you want LSM-like compaction economics but also want bounded, SSD-friendly lookup cost via fractional cascading. Strongly write-optimized.
5. **Reach for a Bw-Tree** when latch contention is the bottleneck (many-core, in-memory-hot workloads) and you're willing to pay the implementation cost of mapping table, delta consolidation, epoch-based GC, and cooperative SMOs. It wins on all three axes but is by far the most complex.
6. **Don't reach for Cache-Oblivious B-Trees yet** — the asymptotics are the same as a well-tuned cache-aware B-Tree and constants and paging effects have kept them academic. Worth revisiting if byte-addressable NVM becomes common.
7. **Buffering and immutability are the two big levers** across the whole design space. Lazy and LA-Tree buffer at the node/subtree level; FD-Trees and Bw-Trees buffer via immutability (append-only runs, append-only deltas). Copy-on-write is immutability without buffering; cache-oblivious is an orthogonal layout trick that can be combined with any of them.

## See Also

- [[01-copy-on-write-b-trees]]
- [[02-lazy-b-trees-and-buffering]]
- [[03-fd-trees]]
- [[04-bw-trees]]
- [[05-cache-oblivious-b-trees]]